# RETRAINING AND EXECUTION SCRIPTS

## RETRAINING SCRIPT

In [3]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor 

# =====================================================================
# 1. MODEL DNA: THE COMPONENT THAT LIVES INSIDE THE PIPELINE
# =====================================================================
class NASAFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Handles structural data cleaning that must be autonomous in the API.
    This ensures the model knows how to read raw text files without headers.
    """
    def __init__(self):
        # Selected sensors based on EDA importance
        self.selected_features = [
            'time_in_cycles', 'sensor_11', 'sensor_4', 'sensor_12',
            'sensor_7', 'sensor_15', 'sensor_21', 'sensor_20'
        ]
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        df = X.copy()
        
        # COLUMN RENAMING:
        # If the input is raw (integer column names), apply the NASA schema.
        if isinstance(df.columns[0], int) or df.columns[0] == 0:
            n_cols = df.shape[1]
            columns = (['unit_number', 'time_in_cycles'] + 
                       [f'op_setting_{i}' for i in range(1, 4)] + 
                       [f'sensor_{i}' for i in range(1, n_cols - 4)])
            df.columns = columns[:n_cols]
            
        # DATA QUALITY & TYPE CASTING:
        # Ensure selected sensors exist, are numeric, and fill nulls (Imputation).
        for col in self.selected_features:
            if col not in df.columns: 
                df[col] = 0.0 
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
            
        # Return only relevant features as a float64 array for the model
        return df[self.selected_features].values.astype(np.float64)

# =====================================================================
# 2. TRAINING & RETRAINING FUNCTION (THE AUTOMATION SCRIPT)
# =====================================================================
def run_model_pipeline(data_path, export_path):
    """
    This function handles the 'Outside' logic (Pandas) to prepare the 
    training set and then trains the 'Inside' logic (Pipeline).
    """
    
    # --- A. DATA INGESTION (OUTSIDE - PANDAS) ---
    # Using sep=r'\s+' to handle whitespace in modern Pandas versions
    df_raw = pd.read_csv(data_path, sep=r'\s+', engine='python', header=None)
    
    # --- B. TARGET CREATION (OUTSIDE - PANDAS) ---
    # The 'y' (RUL) is created here because the Pipeline cannot self-generate targets.
    df_prep = df_raw.copy()
    n_cols = df_prep.shape[1]
    df_prep.columns = (['unit_number', 'time_in_cycles'] + 
                       [f'op_setting_{i}' for i in range(1, 4)] + 
                       [f'sensor_{i}' for i in range(1, n_cols - 4)])

    # Calculate Remaining Useful Life (RUL)
    max_cycles = df_prep.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycles.rename(columns={'time_in_cycles': 'max_cycles'}, inplace=True)
    df_prep = df_prep.merge(max_cycles, on='unit_number')
    
    # y = target variable
    y = df_prep['max_cycles'] - df_prep['time_in_cycles']

    # --- C. PIPELINE ASSEMBLY (THE 'ALUMNO') ---
    # This structure will be saved into the .pkl file
    model_pipeline = Pipeline(steps=[
        ('engineer', NASAFeatureEngineer()),     # Step 1: Handle raw input & renaming
        ('scaler', StandardScaler()),            # Step 2: Normalize values
        ('regressor', XGBRegressor(              # Step 3: Predictive algorithm
            n_estimators=100, 
            learning_rate=0.05, 
            random_state=42
        ))
    ])

    # --- D. TRAINING & DEPLOYMENT ---
    print("Starting training process...")
    model_pipeline.fit(df_raw, y)
    
    # Save the entire Pipeline (DNA) for the API
    joblib.dump(model_pipeline, export_path)
    print(f"Model successfully saved to: {export_path}")

# =====================================================================
# 3. EXECUTION
# =====================================================================
if __name__ == "__main__":
    # Define your local paths
    BASE_PATH = "/Users/rober/cmapss-rul-prediction/"
    INPUT_FILE = os.path.join(BASE_PATH, "02_Data/01_Raw/train_FD001.txt")
    OUTPUT_MODEL = os.path.join(BASE_PATH, "04_Models/nasa_model.pkl")
    
    # Run the retraining/production automation
    run_model_pipeline(INPUT_FILE, OUTPUT_MODEL)

Starting training process...
Model successfully saved to: /Users/rober/cmapss-rul-prediction/04_Models/nasa_model.pkl


## EXECUTION SCRIPT

In [4]:
import pandas as pd
import joblib
import os

# =====================================================================
# 1. ENVIRONMENT & PATHS
# =====================================================================
PROJECT_PATH = '/Users/rober/cmapss-rul-prediction'
VALIDATION_PATH = os.path.join(PROJECT_PATH, '02_Data/02_Validation/validation_FD001.csv')
MODEL_PATH = os.path.join(PROJECT_PATH, '04_Models/nasa_model.pkl')
RESULTS_PATH = os.path.join(PROJECT_PATH, '05_Results/predictions_validation_FD001.csv')

# =====================================================================
# 2. LOAD DATA & MODEL
# =====================================================================
# Load the raw validation data
# Note: The pipeline will handle the feature selection and renaming internally
df_raw = pd.read_csv(VALIDATION_PATH)

# Load the trained autonomous pipeline
# We use joblib as it is standard for scikit-learn pipelines
model_pipeline = joblib.load(MODEL_PATH)

# =====================================================================
# 3. EXECUTION (PREDICTION)
# =====================================================================
# We simply pass the raw dataframe. 
# The 'Inside' logic of the pipeline takes care of the rest.
predictions = model_pipeline.predict(df_raw)

# =====================================================================
# 4. CHECKING PREDICTIONS & SAVING RESULTS
# =====================================================================
# Combine predictions with reference columns for validation
results = pd.DataFrame({
    'unit_number': df_raw['unit_number'],
    'time_in_cycles': df_raw['time_in_cycles'],
    'predicted_RUL': predictions
})

# Sort for better readability
results = results.sort_values(by=['unit_number', 'time_in_cycles']).reset_index(drop=True)

# Display the first 40 rows as a check
print("--- Execution Check: Predicted RUL Samples ---")
display(results.head(40))

# Save the final results for Streamlit or further analysis
results.to_csv(RESULTS_PATH, index=False)
print(f"\nPredictions successfully saved to: {RESULTS_PATH}")

--- Execution Check: Predicted RUL Samples ---


,unit_number,time_in_cycles,predicted_RUL
0,1,1,204.416519
1,1,2,206.447983
2,1,3,208.826538
3,1,4,209.771957
4,1,5,209.215805
5,1,6,203.824295
6,1,7,200.719818
7,1,8,203.603912
8,1,9,198.737930
9,1,10,192.695847



Predictions successfully saved to: /Users/rober/cmapss-rul-prediction/05_Results/predictions_validation_FD001.csv
